In [1]:
#importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df=pd.read_csv("SuperMarket Analysis.csv")
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,...,cogs,gross margin percentage,gross income,Rating,Year,Month,Month Name,Quarter,Year_month,Day_of_week
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,...,522.83,4.761905,26.1415,9.1,2019,1,Jan,1,2019-01,Saturday
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,...,76.40,4.761905,3.8200,9.6,2019,3,Mar,1,2019-03,Friday
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,...,324.31,4.761905,16.2155,7.4,2019,3,Mar,1,2019-03,Sunday
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,...,465.76,4.761905,23.2880,8.4,2019,1,Jan,1,2019-01,Sunday
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,...,604.17,4.761905,30.2085,5.3,2019,2,Feb,1,2019-02,Friday


In [3]:
df.isna().sum()


Invoice ID                 0
Branch                     0
City                       0
Customer type              0
Gender                     0
Product line               0
Unit price                 0
Quantity                   0
Tax 5%                     0
Sales                      0
Date                       0
Time                       0
Payment                    0
cogs                       0
gross margin percentage    0
gross income               0
Rating                     0
Year                       0
Month                      0
Month Name                 0
Quarter                    0
Year_month                 0
Day_of_week                0
dtype: int64

In [4]:
#top 10 products which provide the most revenue
top_products= (df.groupby("Product line")["Sales"].sum().sort_values(ascending=False).head(10))
print(top_products)

Product line
Food and beverages        56144.8440
Sports and travel         55122.8265
Electronic accessories    54337.5315
Fashion accessories       54305.8950
Home and lifestyle        53861.9130
Health and beauty         49193.7390
Name: Sales, dtype: float64


In [7]:
# Which regions perform best and worst?
region_perf = (df.groupby('City')
                 .agg(total_sales=('Sales','sum'), #sum of sales, agg helps to apply multiple operations at once
                      total_orders=('Invoice ID','count'), #number of sales
                      avg_order_value=('Sales','mean')) #average sales amount
                 .sort_values('total_sales', ascending=False))
print(region_perf)

           total_sales  total_orders  avg_order_value
City                                                 
Naypyitaw  110568.7065           328       337.099715
Yangon     106200.3705           340       312.354031
Mandalay   106197.6720           332       319.872506


In [11]:
#analyzing monthly sales trend
monthly=(df.groupby('Year_month')["Sales"].sum().reset_index().sort_values('Year_month'))

In [12]:
print(monthly)


  Year_month       Sales
0    2019-01  116291.868
1    2019-02   97219.374
2    2019-03  109455.507


In [26]:
#customer revenue contribution, Pareto analysis
customer_rev=(df.groupby('Invoice ID')['Sales'].sum().sort_values(ascending=False).reset_index()) #creates a new table with invoice id and sales
customer_rev['cumulative_pct'] = (customer_rev['Sales'].cumsum() /customer_rev['Sales'].sum() * 100)#creates a new column with cumulative sum of revenue per
print(customer_rev.iloc[400:500]) #checking where we reach 80% of the revenue

      Invoice ID     Sales  cumulative_pct
400  220-68-6701  325.3740       71.384366
401  563-91-7120  324.2925       71.484776
402  434-83-9547  323.1480       71.584832
403  796-32-9050  323.0640       71.684862
404  291-21-5991  321.7725       71.784493
..           ...       ...             ...
495  360-39-5055  256.7775       79.906734
496  866-05-7563  256.4100       79.986126
497  195-06-0432  255.1815       80.065138
498  565-67-6697  255.1500       80.144140
499  584-86-7256  254.0160       80.222791

[100 rows x 3 columns]


In [28]:
#checking sales based on customer type
sales_by_type = df.groupby('Customer type').agg(
    total_sales=('Sales', 'sum'),
    total_orders=('Invoice ID', 'count'),
    avg_order_value=('Sales', 'mean')
)

print(sales_by_type)

               total_sales  total_orders  avg_order_value
Customer type                                            
Member          189694.764           565       335.742945
Normal          133271.985           435       306.372379


In [31]:
#Category performance
cat_perf = (df.groupby('Product line')
              .agg(total_sales=('Sales','sum'),
                   avg_profit=('gross income','mean'),
                   order_count=('Invoice ID','count'))
              .sort_values('avg_profit'))
print(cat_perf)

                        total_sales  avg_profit  order_count
Product line                                                
Fashion accessories      54305.8950   14.528062          178
Electronic accessories   54337.5315   15.220597          170
Food and beverages       56144.8440   15.365310          174
Health and beauty        49193.7390   15.411572          152
Sports and travel        55122.8265   15.812630          166
Home and lifestyle       53861.9130   16.030331          160


In [32]:
#summary stats per city
print(df.groupby('City')['Sales'].describe())

           count        mean         std      min         25%        50%  \
City                                                                       
Mandalay   332.0  319.872506  242.447565  18.6375  119.936250  252.88200   
Naypyitaw  328.0  337.099715  263.160870  10.6785  121.474500  271.41975   
Yangon     340.0  312.354031  231.640025  12.6945  137.489625  240.82800   

                  75%      max  
City                            
Mandalay   465.585750  1022.49  
Naypyitaw  495.754875  1042.65  
Yangon     459.340875  1039.29  


In [33]:
# so the above description tells us that 
#Naypyitaw = high-value customers
#Yangon = high-traffic but low-value
#Mandalay = balanced market with few high value customers

In [36]:
# Mean vs median 
print("Mean sales:", round(df['Sales'].mean(), 2))
print("Median sales:", round(df['Sales'].median(), 2))

Mean sales: 322.97
Median sales: 253.85


In [37]:
"""
so we have Right-skewed data (positive skew)
Most customers spend relatively low to medium amounts
A few customers spend very high amounts
Those high values pull the mean upward
"""

'\nso we have Right-skewed data (positive skew)\nMost customers spend relatively low to medium amounts\nA few customers spend very high amounts\nThose high values pull the mean upward\n'

In [38]:
# Month-over-month growth rate
monthly['growth_pct'] = monthly['Sales'].pct_change() * 100
print(monthly[['Year_month','Sales','growth_pct']].tail(12))

  Year_month       Sales  growth_pct
0    2019-01  116291.868         NaN
1    2019-02   97219.374  -16.400540
2    2019-03  109455.507   12.586106


In [39]:
# Pareto analysis — 80/20 rule
top_20_pct = int(len(customer_rev) * 0.20)
top_customers = customer_rev.head(top_20_pct)
revenue_from_top = top_customers['Sales'].sum()
total_revenue = customer_rev['Sales'].sum()
print(f"Top 20% of customers contribute: {revenue_from_top/total_revenue*100:.1f}% of revenue")
# If this is close to 80%, you have a classic Pareto distribution — great insight!

Top 20% of customers contribute: 45.1% of revenue


In [40]:
df.to_csv("SuperMarket Analysis.csv",index=False)

In [41]:
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,...,cogs,gross margin percentage,gross income,Rating,Year,Month,Month Name,Quarter,Year_month,Day_of_week
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,...,522.83,4.761905,26.1415,9.1,2019,1,Jan,1,2019-01,Saturday
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,...,76.40,4.761905,3.8200,9.6,2019,3,Mar,1,2019-03,Friday
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,...,324.31,4.761905,16.2155,7.4,2019,3,Mar,1,2019-03,Sunday
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,...,465.76,4.761905,23.2880,8.4,2019,1,Jan,1,2019-01,Sunday
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,...,604.17,4.761905,30.2085,5.3,2019,2,Feb,1,2019-02,Friday
